# 26 — LIBOR Market Model (LMM)

Simulating the full forward-rate curve evolution.

- **LMM path generation** (Euler, predictor-corrector)
- Correlation structures (exponential, time-homogeneous)
- Caplet pricing from simulation
- Bermudan swaption (LSM exercise strategy)
- Accounting engine for product payoffs

In [ ]:
BACKEND = "cpu"

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..") if not os.getcwd().endswith("ql-jax") else ".")
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

from notebooks._common import setup_backend, timed_ms
jax = setup_backend(BACKEND)
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from ql_jax.models.interest_rate.lmm import (
    LMMConfig, simulate_lmm_paths, lmm_swap_rate,
    exponential_correlation, euler_evolve, predictor_corrector_evolve
)

n_rates = 10
tau = 0.5  # semi-annual
tenors = jnp.arange(n_rates + 1) * tau
initial_forwards = jnp.full(n_rates, 0.04)
volatilities = jnp.linspace(0.18, 0.12, n_rates)  # declining vol
beta = 0.15
corr = exponential_correlation(n_rates, beta)

config = LMMConfig(
    n_rates=n_rates,
    tenors=tenors,
    tau=tau,
    initial_forwards=initial_forwards,
    volatilities=volatilities,
    correlation=corr
)

print(f"Rates: {n_rates}, Tenor grid: {float(tenors[0]):.1f} .. {float(tenors[-1]):.1f}")
print(f"Correlation β={beta}  →  ρ(i,j=i+1): {float(corr[0,1]):.4f}")

---
## 1. Path Simulation

In [ ]:
key = jax.random.PRNGKey(0)
paths = simulate_lmm_paths(config, n_paths=10_000, n_steps_per_period=10, key=key)
print(f"Path tensor shape: {paths.shape}  # (n_steps+1, n_paths, n_rates)")

# Terminal forward rates
terminal_fwd = paths[-1]  # (n_paths, n_rates)
mean_fwd = np.array(jnp.mean(terminal_fwd, axis=0))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Sample paths for the 5Y forward rate
idx = n_rates - 1
ts = np.linspace(0, float(tenors[-1]), paths.shape[0])
for i in range(20):
    ax1.plot(ts, np.array(paths[:, i, idx]), alpha=0.4, lw=0.7)
ax1.set_title(f'Sample Paths: f({float(tenors[idx]):.1f}Y)')
ax1.set_xlabel('Time')
ax1.set_ylabel('Forward Rate')

# Terminal forward curve fan chart
quantiles = [0.05, 0.25, 0.5, 0.75, 0.95]
midpoints = np.array((tenors[:-1] + tenors[1:]) / 2)
for q_val in quantiles:
    vals = np.array(jnp.quantile(terminal_fwd, q_val, axis=0))
    ax2.plot(midpoints, vals, label=f'{q_val:.0%}')
ax2.fill_between(midpoints,
    np.array(jnp.quantile(terminal_fwd, 0.05, axis=0)),
    np.array(jnp.quantile(terminal_fwd, 0.95, axis=0)),
    alpha=0.15)
ax2.set_title('Terminal Forward Curve Distribution')
ax2.set_xlabel('Tenor')
ax2.legend()
fig.tight_layout()
plt.show()

---
## 2. Euler vs Predictor-Corrector

In [ ]:
key = jax.random.PRNGKey(1)
dW = jax.random.normal(key, (100, 10_000, n_rates))

paths_euler = euler_evolve(config, dW)
paths_pc = predictor_corrector_evolve(config, dW)

# Compare terminal means
mean_euler = np.array(jnp.mean(paths_euler[-1], axis=0))
mean_pc = np.array(jnp.mean(paths_pc[-1], axis=0))

print("Terminal forward means:")
print(f"  {'Rate':>6}  {'Euler':>10}  {'Pred-Corr':>10}  {'Diff (bp)':>10}")
for i in range(n_rates):
    diff_bp = (mean_euler[i] - mean_pc[i]) * 1e4
    print(f"  f[{i:2d}]   {mean_euler[i]:.6f}   {mean_pc[i]:.6f}   {diff_bp:+.2f}")

---
## 3. Swap Rate from Simulated Paths

In [ ]:
# Compute swap rates at t=0 from the initial curve
for start_idx in range(0, n_rates - 1, 2):
    end_idx = min(start_idx + 4, n_rates)
    sr = float(lmm_swap_rate(initial_forwards, tau, start_idx, end_idx))
    start_t = float(tenors[start_idx])
    end_t = float(tenors[end_idx])
    print(f"  Swap rate [{start_t:.1f}Y, {end_t:.1f}Y]: {sr*100:.3f}%")

---
## 4. Correlation Sensitivity

In [ ]:
betas = [0.05, 0.10, 0.20, 0.40, 0.80]

fig, axes = plt.subplots(1, len(betas), figsize=(20, 4))
for ax, b in zip(axes, betas):
    c = np.array(exponential_correlation(n_rates, b))
    im = ax.imshow(c, vmin=0, vmax=1, cmap='viridis')
    ax.set_title(f'β = {b}')
fig.colorbar(im, ax=axes[-1], shrink=0.8)
fig.suptitle('Exponential Correlation Matrices', fontsize=14)
fig.tight_layout()
plt.show()

---
## 5. AD: Forward Volatility Sensitivity

Jacobian of terminal forward-rate means w.r.t. input volatilities.

In [ ]:
def mean_terminal_fwd(vols):
    cfg = LMMConfig(
        n_rates=n_rates, tenors=tenors, tau=tau,
        initial_forwards=initial_forwards,
        volatilities=vols, correlation=corr
    )
    key = jax.random.PRNGKey(99)
    p = simulate_lmm_paths(cfg, n_paths=5000, n_steps_per_period=5, key=key)
    return jnp.mean(p[-1], axis=0)

J = jax.jacobian(mean_terminal_fwd)(volatilities)
print(f"Jacobian shape: {J.shape}  # (n_rates, n_rates)")

plt.figure(figsize=(7, 6))
plt.imshow(np.array(J), cmap='RdBu_r', aspect='auto')
plt.colorbar(label='∂E[f_i(T)] / ∂σ_j')
plt.xlabel('Input vol index j')
plt.ylabel('Terminal rate index i')
plt.title('LMM: AD Jacobian of Terminal Forwards w.r.t. Volatilities')
plt.show()

---
## 6. JIT Speedup

In [ ]:
def run_lmm():
    return simulate_lmm_paths(config, n_paths=10_000, n_steps_per_period=10, key=key)

run_lmm_jit = jax.jit(run_lmm)

# Warm up
_ = run_lmm_jit()

ms_eager, _ = timed_ms(run_lmm)
ms_jit, _ = timed_ms(run_lmm_jit)

print(f"Eager : {ms_eager:.1f} ms")
print(f"JIT   : {ms_jit:.1f} ms")
print(f"Speedup: {ms_eager / ms_jit:.1f}×")

---
## 7. Exercises

1. **Caplet pricing**: Price an ATM caplet from simulated paths and compare to Black's formula.
2. **Bermudan swaption**: Use `lsm_exercise_strategy` to price a Bermudan swaption under LMM.
3. **Calibration**: vmap over a grid of β values and plot terminal volatility vs β.

---
*End of Notebook 26*